In [1]:


import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Updated LGBM Model
lgb_params = {
    'max_depth': 5,
    'min_child_samples': 1,
    'learning_rate': 0.06545162204587239,
    'n_estimators': 927,
    'subsample': 0.981043589736463,
    'colsample_bytree': 0.4120165727598161,
    'reg_alpha': 0.05200005749596763,
    'reg_lambda': 0.970699319910319
}
lgb_model = lgb.LGBMClassifier(**lgb_params)

# Updated XGBoost Model
xgb_params = {
    'max_depth': 5,
    'min_child_weight': 10,
    'learning_rate': 0.06412082210309321,
    'n_estimators': 796,
    'subsample': 0.9631084457336389,
    'colsample_bytree': 0.5501821632318996,
    'random_state': 42
}
xgb_model = XGBClassifier(**xgb_params)

# Updated CatBoost Model
cat_params = {
    'iterations': 839,
    'depth': 5,
    'min_data_in_leaf': 4,
    'learning_rate': 0.08678125515120433,
    'verbose': 0
}
cat_model = CatBoostClassifier(**cat_params)



**Finding Optimal Weights through Optuna**

In [ ]:
# Define objective function for Optuna to optimize
X = train_data_combined.drop('Exited', axis=1)
y = train_data_combined['Exited']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

xgb_model.fit(X_train[[feature for feature in xgb_selected_features if feature != 'Exited']], y_train, verbose=0)
lgb_model.fit(X_train[[feature for feature in lgb_selected_features if feature != 'Exited']], y_train, verbose=0)
cat_model.fit(X_train[[feature for feature in cb_selected_features if feature != 'Exited']], y_train, verbose=0)

def objective(trial):
    xgb_weight = trial.suggest_uniform('xgb_weight', 0, 1)
    lgb_weight = trial.suggest_uniform('lgb_weight', 0, 1)
    cat_weight = trial.suggest_uniform('cat_weight', 0, 1)

    # Normalize weights to sum up to 1
    total_weight = xgb_weight + lgb_weight + cat_weight
    xgb_weight /= total_weight
    lgb_weight /= total_weight
    cat_weight /= total_weight

    # Ensemble predictions
    ensemble_pred_proba = (
        xgb_weight * xgb_model.predict_proba(X_test[[feature for feature in xgb_selected_features if feature != 'Exited']])[:, 1] +
        lgb_weight * lgb_model.predict_proba(X_test[[feature for feature in lgb_selected_features if feature != 'Exited']])[:, 1] +
        cat_weight * cat_model.predict_proba(X_test[[feature for feature in cb_selected_features if feature != 'Exited']])[:, 1]
    )

    # Assuming y_test is available
    auc_score = roc_auc_score(y_test, ensemble_pred_proba)

    return auc_score

# Optimize using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

In [ ]:
# Get the best weights
best_weights = study.best_params
xgb_weight = best_weights['xgb_weight']
lgb_weight = best_weights['lgb_weight']
cat_weight = best_weights['cat_weight']

total_weight = xgb_weight + lgb_weight + cat_weight
xgb_weight /= total_weight
lgb_weight /= total_weight
cat_weight /= total_weight

print('xgb_weight: ',xgb_weight)
print('lgb_weight: ',lgb_weight)
print('cat_weight: ',cat_weight

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier

y = train_data_combined['Exited']

xgb_model.fit(train_data_combined[xgb_selected_features].drop('Exited', axis=1), y)
xgb_pred_proba = xgb_model.predict_proba(test_data_combined[[feature for feature in xgb_selected_features if feature != 'Exited']])[:, 1]

lgb_model.fit(train_data_combined[lgb_selected_features].drop('Exited', axis=1), y)
lgb_pred_proba = lgb_model.predict_proba(test_data_combined[[feature for feature in lgb_selected_features if feature != 'Exited']])[:, 1]

cat_model.fit(train_data_combined[cb_selected_features].drop('Exited', axis=1), y)
cb_pred_proba = cat_model.predict_proba(test_data_combined[[feature for feature in cb_selected_features if feature != 'Exited']])[:, 1]

#ensemble_pred_proba = (xgb_pred_proba * 0) + (lgb_pred_proba * 0) + (cb_pred_proba * 1) 
ensemble_pred_proba = (xgb_pred_proba * xgb_weight) + (lgb_pred_proba * lgb_weight) + (cb_pred_proba * cat_weight) 
#ensemble_pred_proba = np.mean([xgb_pred_proba, lgb_pred_proba, cb_pred_proba], axis=0)

# Assuming 'test_data_combined' is the DataFrame for the test set
ensemble_submission_df = pd.DataFrame({
    'id': id_test,
    'Exited': ensemble_pred_proba  # Fill in the predicted probabilities
})

# Save the submission DataFrame to a CSV file
ensemble_submission_df.to_csv('submission.csv', index=False)

ensemble_submission_df.head(10)

# rank ensembling.
<mark>this is good to improve AUC</mark>


Remember: competitions using ROC AUC only care about **ranking**, not the exact probability values.

---

# 1️⃣ What AUC actually checks

AUC only checks:

> **Are positive samples ranked higher than negative samples?**

Example:

| Sample | True label | Prediction |
| ------ | ---------- | ---------- |
| A      | 1          | 0.9        |
| B      | 0          | 0.6        |
| C      | 1          | 0.5        |
| D      | 0          | 0.2        |

Model ranking:

```
A > B > C > D
```

But the correct ranking should be:

```
A > C > B > D
```

Because **C is positive but ranked below B**.

AUC penalizes that mistake.

---

# 2️⃣ Why we combine multiple models

Suppose you train two models.

### Model 1

| Sample | Prediction |
| ------ | ---------- |
| A      | 0.9        |
| B      | 0.7        |
| C      | 0.4        |
| D      | 0.2        |

Ranking:

```
A > B > C > D
```

Mistake: **B ranked above C**

---

### Model 2

| Sample | Prediction |
| ------ | ---------- |
| A      | 0.8        |
| B      | 0.3        |
| C      | 0.6        |
| D      | 0.1        |

Ranking:

```
A > C > B > D
```

This model fixes the mistake.

---

# 3️⃣ If we average probabilities

Normal ensembling:

```
(A): (0.9 + 0.8)/2 = 0.85
(B): (0.7 + 0.3)/2 = 0.50
(C): (0.4 + 0.6)/2 = 0.50
(D): (0.2 + 0.1)/2 = 0.15
```

Ranking becomes:

```
A > B ≈ C > D
```

Not great.

---

# 4️⃣ Rank averaging (the trick)

Instead of averaging probabilities, we average **ranks**.

### Model 1 ranks

| Sample | Rank |
| ------ | ---- |
| A      | 4    |
| B      | 3    |
| C      | 2    |
| D      | 1    |

---

### Model 2 ranks

| Sample | Rank |
| ------ | ---- |
| A      | 4    |
| C      | 3    |
| B      | 2    |
| D      | 1    |

---

### Average ranks

| Sample | Avg rank |
| ------ | -------- |
| A      | 4        |
| C      | 2.5      |
| B      | 2.5      |
| D      | 1        |

Final ranking:

```
A > C ≈ B > D
```

Now **C moves higher**, improving ranking.

---

# 5️⃣ Why this works

Different models make **different ranking mistakes**.

Rank averaging:

* reduces noise
* keeps consistent ordering
* improves AUC stability

This is why top competitors on Kaggle often ensemble **10–30 models**.

---

# 6️⃣ Practical Python example

```python
from scipy.stats import rankdata
import numpy as np

rank1 = rankdata(pred_model1)
rank2 = rankdata(pred_model2)

final_prediction = (rank1 + rank2) / 2
```

Submit `final_prediction`.

---

# 7️⃣ Important detail

Rank averaging is especially useful when models have **different scales**.

Example:

| Model          | Output scale |
| -------------- | ------------ |
| LightGBM       | 0–1          |
| Neural network | 0–0.2        |
| SVM            | -3 to 3      |

Averaging probabilities would be bad.

But **ranks normalize everything automatically**.

---

✅ **Simple intuition**

Instead of averaging **scores**, we average **positions in the ranking**.

---

If you want, I can also explain a **very powerful Kaggle technique called "Out-of-Fold (OOF) stacking"**.

That method is used in **almost every winning solution** and works perfectly with what you're already doing with Optuna + LightGBM.
